# Khóa luận GraphRAG — Notebook làm việc trên Kaggle

**Cấu hình bắt buộc (panel phải):** Accelerator `GPU T4 x2` · Internet `On` · Persistence `Files only` · Visibility `Private`

## Cái gì sống sót qua restart / đóng session

| Sống sót — `/kaggle/working/` | Bị xóa |
|---|---|
| `wheels/*.whl` (122 MB, CUDA) | Mọi `pip install` |
| `llamacpp-b10165-cpu-static-x64.tar.gz` | `/tmp/*` (mã nguồn + `build/`) |
| `*.sha256`, `base_manifest.json`, `repo_id.txt` | Cache HF (GGUF 2,5 GB) |
| Kaggle Secrets | `os.environ`, mọi biến Python |

## Giá trị đã ghim — tuyên bố tái lập

```
Python            3.12.13          torch             2.10.0+cu128
transformers      5.5.0            trl               0.24.0
peft              0.20.0           unsloth           2026.7.5
huggingface_hub   1.25.1

llama.cpp tag     b10165
llama.cpp commit  81616410050cb5d8b733b39863807f4852591d8a
tarball sha256    9f8e92b8a69b3c8399e6f42324430f8a0faaf1bba707deeee7c8cb96bbd9c6d5
llama-cpp-python  0.3.16 (cp312, CUDA 75;89)
wheel sha256      a3cb84bddb15c1759a0ece5ec8ef9d10d1419f926c895064d1a91ec517fd0da7
```

---
# PHẦN A — Khởi động (chạy mỗi phiên)

### A1 · Biến môi trường — PHẢI là cell đầu tiên, trước mọi `import torch`

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"]    = "0"    # Kaggle cấp 2×T4; ép 1 GPU để giữ tính tái lập
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"    # hub v1.x: hf_transfer đã bị gỡ
print("OK")

### A2 · Kiểm những gì sống sót

In [ ]:
import glob, pathlib
for pat in ["wheels/*.whl", "llamacpp-*-static-*.tar.gz", "*.sha256",
            "base_manifest.json", "repo_id.txt"]:
    hits = glob.glob(f"/kaggle/working/{pat}")
    print(f"{pat:32s}", [pathlib.Path(h).name for h in hits] or "THIẾU")

### A3 · Cài đặt — MỘT lượt duy nhất

Nguyên tắc rút ra từ nhiều lần vấp: cài hết trong một lượt, kiểm phiên bản một lần, rồi
không `pip install` gì nữa cho tới hết phiên. Mỗi lần cài thêm là một cơ hội để torch bị
đổi sang bản CPU-only.

In [ ]:
!pip install -q "huggingface_hub==1.25.1" hf_xet
!pip install -q /kaggle/working/wheels/llama_cpp_python-*.whl
!pip install -q "transformers==5.5.0" "trl==0.24.0" "peft==0.20.0" "unsloth==2026.7.5" \
                accelerate bitsandbytes datasets sentencepiece protobuf safetensors

### A4 · Cổng chặn — nếu dòng nào sai, DỪNG và sửa trước khi đi tiếp

In [ ]:
!python -c "import torch, huggingface_hub as h, transformers as t, trl; \
print('torch          :', torch.__version__, '| cuda:', torch.cuda.is_available()); \
print('huggingface_hub:', h.__version__); \
print('transformers   :', t.__version__); \
print('trl            :', trl.__version__); \
assert torch.cuda.is_available(), 'torch MẤT CUDA'; \
assert not torch.__version__.endswith('+cpu'), 'torch bản CPU-only'; \
print('>>> CỔNG CHẶN: QUA')"

**Nếu torch ra `+cpu`:** chạy ô sửa chữa dưới đây rồi **Run → Restart session**,
sau đó chạy lại từ A1.

In [ ]:
# CHỈ chạy khi A4 báo lỗi torch
!pip install -q torch==2.10.0 torchvision==0.25.0 torchaudio==2.10.0 \
    --index-url https://download.pytorch.org/whl/cu128

### A5 · Đăng nhập HF (token trong `os.environ` mất theo restart)

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, whoami

HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

HF_REPO = open("/kaggle/working/repo_id.txt").read().strip()
print("HF:", whoami()["name"], "| kho:", HF_REPO)

---
# PHẦN B — Repo GraphRAG

### B1 · Clone

Sửa `GH_URL` thành repo của bạn. Nếu repo **private**, dùng dạng có token:
`https://<PAT>@github.com/<user>/<repo>.git` và đặt PAT vào Kaggle Secrets, đừng dán thẳng.

In [ ]:
GH_URL    = "https://github.com/<user>/vn-legal-graphrag.git"   # <== SỬA
GH_BRANCH = "main"
REPO_DIR  = "/kaggle/working/vn-legal-graphrag"

import os, subprocess
if os.path.exists(REPO_DIR):
    subprocess.run(f"cd {REPO_DIR} && git fetch --all && git checkout {GH_BRANCH} "
                   f"&& git pull", shell=True, check=True)
else:
    subprocess.run(f"git clone --branch {GH_BRANCH} {GH_URL} {REPO_DIR}",
                   shell=True, check=True)

print(subprocess.run(f"cd {REPO_DIR} && git log -1 --format='commit %H%n  %s%n  %ci'",
                     shell=True, capture_output=True, text=True).stdout)
print(">>> GHI COMMIT HASH NÀY VÀO KHÓA LUẬN — nó ghim phiên bản CODE")

### B2 · ĐIỂM DỪNG — đọc README trước khi chạy bất cứ lệnh nào

Nếu README có bước `pip install -r requirements.txt`, **đừng chạy ngay**. Đối chiếu với bộ
đã ghim ở đầu notebook. Một `pip install` sai chỗ có thể hạ torch về bản CPU và bạn sẽ
phát hiện điều đó sau sáu giờ huấn luyện, hoặc tệ hơn, trên máy trả tiền.

In [ ]:
!echo "===================== finetune/README.md ====================="
!cat {REPO_DIR}/finetune/README.md
!echo ""
!echo "===================== các file phụ thuộc ====================="
!ls -la {REPO_DIR}
!for f in requirements.txt pyproject.toml finetune/requirements.txt; do \
   [ -f {REPO_DIR}/$f ] && echo "--- $f ---" && cat {REPO_DIR}/$f; done

### B3 · Kiểm import

In [ ]:
import subprocess
r = subprocess.run(
    'python -c "from finetune.replay import build_chat_messages; print(\'OK\')"',
    shell=True, cwd=REPO_DIR, capture_output=True, text=True)
print(r.stdout or r.stderr)

Nếu báo `ModuleNotFoundError: finetune`, thiếu `__init__.py` hoặc phải chạy từ gốc repo.
Cách chắc chắn hơn là đặt gốc repo vào `PYTHONPATH`:

In [ ]:
import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
import os; os.environ["PYTHONPATH"] = REPO_DIR
from finetune.replay import build_chat_messages
print("OK — import được từ trong kernel")

### B4 · Preflight — ba đường dẫn phải tồn tại

Kiểm trước khi chạy. Thất bại ở đây tốn 10 giây; thất bại giữa chừng lệnh 1 tốn nhiều hơn.

In [ ]:
import pathlib

GGUF_REL = "finetune/models/Qwen3-4B-Instruct-2507-Q4_K_M.gguf"
SRC_REL  = "data/evaluation/results_graphrag_20260710-085236.json"
IDS_REL  = "finetune/data/gate_ids.json"

for rel in [SRC_REL, IDS_REL, GGUF_REL]:
    p = pathlib.Path(REPO_DIR, rel)
    size = f"  ({p.stat().st_size/1e9:.2f} GB)" if p.exists() and p.stat().st_size > 1e8 else ""
    print(("CÓ    " if p.exists() else "THIẾU ") + rel + size)

### B5 · Nối file GGUF từ cache HF

Symlink thay vì copy: file 2,5 GB, `/kaggle/working` chỉ có 20 GB. Cũng xử lý luôn việc đổi tên
(bartowski dùng tiền tố `Qwen_`, repo mong đợi tên không tiền tố).

In [ ]:
import json, os, pathlib

m   = json.loads(pathlib.Path("/kaggle/working/base_manifest.json").read_text())
dst = pathlib.Path(REPO_DIR, GGUF_REL)
dst.parent.mkdir(parents=True, exist_ok=True)
if dst.exists() or dst.is_symlink():
    dst.unlink()
os.symlink(m["path"], dst)

pathlib.Path(REPO_DIR, "finetune/results").mkdir(parents=True, exist_ok=True)

print("symlink :", dst)
print("     -> :", m["path"])
print("sha256  :", m["sha256"])
print("kích thước:", round(dst.stat().st_size/1e9, 2), "GB")

### B6 · Kiểm tham số sinh trong `replay.py`

README viết *"cùng seed"* nhưng không lệnh nào truyền `--seed`. Phải xác nhận `replay.py` ghim
seed bên trong. Nếu seed thả nổi thì bốn ô của ma trận không so sánh được với nhau — đúng loại
confound README cảnh báo ở dòng cuối.

In [ ]:
!grep -n 'seed\|temperature\|top_p\|top_k\|min_p\|presence_penalty\|max_tokens\|n_ctx\|n_gpu_layers' \
    {REPO_DIR}/finetune/replay.py | head -30

Đối chiếu với bộ đã chốt: `temperature=0.7  top_p=0.8  top_k=20  min_p=0`,
`n_ctx=16384`, `n_gpu_layers=-1`, seed cố định. Thiếu hoặc lệch chỗ nào thì sửa **trước** khi chạy —
chạy lại bốn lệnh tốn 40 phút.

### B7 · Đặt biến môi trường

Bằng Python chứ không `export`: mỗi ô `!` sinh một shell mới, `export` không sống qua ô kế tiếp.
Các ô `!` kế thừa `os.environ` nên cách này chắc chắn.

In [ ]:
import json, os, pathlib

os.environ["GGUF"] = GGUF_REL
os.environ["SRC"]  = SRC_REL

ids = json.load(open(pathlib.Path(REPO_DIR, IDS_REL), encoding="utf-8"))["ids_csv"]
os.environ["IDS"]  = ids

print("số câu :", len(ids.split(",")), "(kỳ vọng 15)")
print("IDS    :", ids[:100], "...")

---
### B8 · Bốn lệnh

Chạy **từng ô một**, đọc output trước khi sang ô kế. Thứ tự đã sắp có chủ ý: lệnh 1
(zero-shot) là đường an toàn nhất và xác thực toàn bộ đường ống trước khi thử nhánh
few-shot vốn có nguy cơ vượt trần 16k.

Bốn ô này là ma trận: `{n-shot 0, 2} × {presence_penalty 1.0, 0}`. Mọi tham số khác giữ
nguyên tuyệt đối.

In [ ]:
%cd {REPO_DIR}

**Lệnh 1** — zero-shot, `presence_penalty=1.0` · *đường an toàn nhất, chạy trước*

In [ ]:
!python -m finetune.replay --input "$SRC" --model "$GGUF" --ids "$IDS" \
  --n-shot 0 --presence-penalty 1.0 --tag gate-s0-pp10 \
  --dump-prompt finetune/results/prompt_gate-s0-pp10.txt

**Lệnh 2** — few-shot 2 ví dụ, `presence_penalty=1.0` · *nhánh rủi ro nhất*

Nếu ném `ValueError: Requested tokens (N) exceed context window of 16384`, nguyên nhân là ví dụ
few-shot mang theo khối ngữ cảnh riêng. Cách sửa: dùng exemplar **chỉ có câu hỏi + đáp án mẫu,
không kèm ngữ cảnh** (~350 token/ví dụ thay vì 5k). Bạn đang dạy định dạng đầu ra, không dạy nội dung.

In [ ]:
!python -m finetune.replay --input "$SRC" --model "$GGUF" --ids "$IDS" \
  --n-shot 2 --presence-penalty 1.0 --tag gate-s2-pp10 \
  --dump-prompt finetune/results/prompt_gate-s2-pp10.txt

**Lệnh 3** — zero-shot, `presence_penalty=0`

In [ ]:
!python -m finetune.replay --input "$SRC" --model "$GGUF" --ids "$IDS" \
  --n-shot 0 --presence-penalty 0 --tag gate-s0-pp00

**Lệnh 4** — few-shot 2 ví dụ, `presence_penalty=0`

In [ ]:
!python -m finetune.replay --input "$SRC" --model "$GGUF" --ids "$IDS" \
  --n-shot 2 --presence-penalty 0 --tag gate-s2-pp00

### B9 · Đo prompt đã dump

Lệnh 1 và 2 ghi ra prompt đã render. Đây là dữ liệu để (a) xác nhận có vượt trần không,
và (b) làm vế trái của phép so offline ↔ runtime — đối chiếu con số dưới đây với
`usage.prompt_tokens` mà `replay.py` báo.

Nhớ bất đối xứng: đường đánh giá dùng `add_generation_prompt=True`, đường huấn luyện dùng
`False`. Không khớp sẽ tạo lệch hằng số ~3 token trông y hệt lỗi template.

In [ ]:
from transformers import AutoTokenizer
import glob, pathlib

tk = AutoTokenizer.from_pretrained("Qwen/Qwen3-4B-Instruct-2507")
for f in sorted(glob.glob(f"{REPO_DIR}/finetune/results/prompt_gate-*.txt")):
    n = len(tk(pathlib.Path(f).read_text(encoding="utf-8"), add_special_tokens=False)["input_ids"])
    flag = "  <== VƯỢT TRẦN 16384" if n > 16384 else ""
    print(f"{pathlib.Path(f).name:34s} {n:7,} token{flag}")

### B10 · Bốn chỉ báo nhị phân

Cổng quyết định không phải một con số phần trăm — 15 câu không phân giải nổi 10% với 30%.
Nó phân giải được **kiểu hỏng**, và đó mới là thứ quyết định có phải leo lên 8B hay không.

| Chỉ báo | Định nghĩa |
|---|---|
| `no_loop` | không chạm `max_tokens` |
| `format_ok` | khối trích dẫn parse được |
| `id_verbatim` | slug chép đúng nguyên văn, không bịa |
| `in_context` | điều luật trích dẫn **có mặt trong ngữ cảnh đưa vào** |

| Quan sát | Chẩn đoán | Quyết định |
|---|---|---|
| `no_loop` thấp | lỗi tham số sinh | sửa `presence_penalty` rồi đo lại — **chưa kết luận gì** |
| `in_context` cao, `format_ok` thấp | hiểu được, chưa biết trình bày | **giữ 4B** — đúng thứ fine-tune sửa |
| `format_ok` cao, `in_context` thấp | không truy hồi được ở ngữ cảnh dài | lý do chính đáng duy nhất để thử 8B |
| cả hai đều thấp | bài toán khó hơn dự kiến | giữ 4B, viết nhánh "tinh chỉnh là điều kiện cần" |

**Chọn `presence_penalty` nhỏ nhất mà `no_loop` = 100%.** Nếu `pp=0` đã đạt, dùng `pp=0` —
không có lý do gì phạt chính hành vi chép nguyên văn mà bạn cần.

---
# PHẦN C — Khôi phục hạ tầng (chỉ chạy khi cần)

### C1 · `llama-quantize` + mã nguồn `convert_hf_to_gguf.py`

Cần cho bước export GGUF. Khôi phục từ tarball đã xác minh, **không build lại** — build lại
sinh sha256 mới và làm sai ghi chép trong khóa luận.

In [ ]:
%%bash
set -e
TAG=b10165
BIN=/tmp/dry/llama.cpp/build/bin

cd /kaggle/working && sha256sum -c llamacpp-$TAG.sha256

[ -d /tmp/dry/llama.cpp ] || git clone --depth 1 --branch $TAG \
    https://github.com/ggml-org/llama.cpp /tmp/dry/llama.cpp
mkdir -p $BIN && tar xzf /kaggle/working/llamacpp-$TAG-cpu-static-x64.tar.gz -C $BIN
chmod +x $BIN/llama-* 2>/dev/null || true

# gguf-py khớp ĐÚNG commit đã ghim. KHÔNG dùng requirements-convert_hf_to_gguf.txt:
# file đó có --extra-index-url .../whl/cpu và sẽ hạ torch về bản CPU-only.
pip install -q /tmp/dry/llama.cpp/gguf-py

$BIN/llama-quantize --help > /dev/null 2>&1 && echo ">>> llama-quantize chạy được"

### C2 · Tải lại model gốc

Cache HF bị xóa theo session. `path` trong `base_manifest.json` đã hỏng nhưng file JSON vẫn
nằm đó — đây là bẫy: `Llama(model_path=...)` báo file not found dù manifest trông bình thường.

`assert` dưới đây chứng minh file hôm nay giống hệt file hôm qua — đó là lý do ta ghim `revision`.

In [ ]:
from huggingface_hub import hf_hub_download
import hashlib, pathlib, json

MF  = pathlib.Path("/kaggle/working/base_manifest.json")
old = json.loads(MF.read_text())
p   = hf_hub_download(old["repo"], old["file"], revision=old["revision"])
h   = hashlib.sha256(pathlib.Path(p).read_bytes()).hexdigest()
assert h == old["sha256"], "sha256 LỆCH — file trên Hub đã đổi!"

old["path"] = p
MF.write_text(json.dumps(old, indent=2))
print("OK, sha256 khớp. path mới:", p)

### C3 · Smoke test GPU (30 giây)

Mốc so sánh đã đo: **35,8 tok/s** ở prompt ngắn, **30,3 tok/s** ở prompt ~10k
(prefill 1.267 tok/s). Log phải có `offloaded N/N layers to GPU` — nếu là `M/N` thì
tràn VRAM, vài layer rơi xuống CPU và tính tái lập mất.

In [ ]:
import json, pathlib, time
from llama_cpp import Llama

m   = json.loads(pathlib.Path("/kaggle/working/base_manifest.json").read_text())
llm = Llama(model_path=m["path"], n_ctx=16384, n_gpu_layers=-1,
            n_batch=2048, seed=42, verbose=True)

t0 = time.time()
o = llm.create_chat_completion(
    messages=[{"role": "user", "content": "Tốc độ tối đa trong khu dân cư?"}],
    temperature=0.7, top_p=0.8, top_k=20, min_p=0.0,
    presence_penalty=1.0, seed=42, max_tokens=128)
n = o["usage"]["completion_tokens"]
print(f"\n{n/(time.time()-t0):.1f} tok/s   (mốc: 35.8)")

**Giải phóng VRAM trước khi huấn luyện** — `llm` giữ ~5 GB, T4 chỉ có 14,5 GB.

In [ ]:
try:
    del llm
except NameError:
    pass
import gc, torch; gc.collect(); torch.cuda.empty_cache()
print("VRAM trống:", round(torch.cuda.mem_get_info()[0]/1e9, 1), "GB")